# 🎬 Atik's Addaverse - Cloud Video Streamer
### Direct Google Drive to YouTube Cloud Uploader (0MB Local Download)
*Inspired by [@Addaverse_atik](https://www.youtube.com/@Addaverse_atik)*

**Features:**
- ⚡ Streams directly from Google Drive to your YouTube channel at Google cloud speed (~500 MB/s)
- 📱 0 MB downloaded to your computer or phone
- 🔒 100% Free, zero credit card, zero API console setup
- 🎯 Gives you a direct link to open the draft in **YouTube Studio**

### Step 1: Install Required Cloud Libraries

In [ ]:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib gdown -q
print('✅ Cloud libraries installed!')

### Step 2: Sign in to YouTube (1-Click Google Sign-In)

In [ ]:
from google.colab import auth
import os
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# Authenticate directly with Google in 1 click
print('🔐 Click the link below to sign in to your YouTube channel account:')
auth.authenticate_user()

### Step 3: Stream Video from Google Drive & Upload to YouTube

In [ ]:
import re
import gdown
from googleapiclient.discovery import build
from google.colab import auth

# @title 🎬 Enter Video Details
google_drive_link = "https://drive.google.com/file/d/.../view" # @param {type:"string"}
video_title = "My Addaverse Video" # @param {type:"string"}
privacy_status = "private" # @param ["private", "unlisted", "public"]

# Extract file id
match = re.search(r'[-\w]{25,}', google_drive_link)
if not match:
    raise ValueError("❌ Invalid Google Drive link! Please paste a valid link.")

file_id = match.group(0)
print(f"📁 Found Drive File ID: {file_id}")

# Download directly to Colab's fast cloud RAM (Not your computer!)
output_file = "cloud_video.mp4"
print("⚡ Pulling video into Google Cloud RAM...")
gdown.download(id=file_id, output=output_file, quiet=False)

# Upload to YouTube using Authenticated Credentials
from google.auth import default
creds, _ = default(scopes=["https://www.googleapis.com/auth/youtube.upload"])
youtube = build("youtube", "v3", credentials=creds)

body = {
    "snippet": {
        "title": video_title,
        "description": "Uploaded via Atik's Addaverse Cloud Streamer directly from Google Drive.",
        "tags": ["Addaverse", "Atik"]
    },
    "status": {
        "privacyStatus": privacy_status
    }
}

media = MediaFileUpload(output_file, chunksize=-1, resumable=True)

print("🚀 Streaming to YouTube...")
request = youtube.videos().insert(
    part="snippet,status",
    body=body,
    media_body=media
)

response = None
while response is None:
    status, response = request.next_chunk()
    if status:
        print(f"Uploading: {int(status.progress() * 100)}%")

video_id = response['id']
studio_url = f"https://studio.youtube.com/video/{video_id}/edit"
watch_url = f"https://youtu.be/{video_id}"

print("\n" + "="*60)
print("🎉 SUCCESS! Video uploaded to YouTube!")
print(f"🎬 Studio Edit Link: {studio_url}")
print(f"📺 Watch Link: {watch_url}")
print("="*60)

# Clean up Colab temporary storage
if os.path.exists(output_file):
    os.remove(output_file)
